# GeoJSON lihtsustamine

Lae üles oma .geojson failid, skript teeb need väiksemaks.

In [ ]:
!pip install geopandas -q

In [ ]:
from google.colab import files
uploaded = files.upload()  # lae üles kõik .geojson failid

In [ ]:
import geopandas as gpd
import os

geojson_failid = [f for f in os.listdir('.') if f.endswith('.geojson')]
print('Leitud failid:', geojson_failid)

for fail in geojson_failid:
    suurus_enne = os.path.getsize(fail) / 1024 / 1024
    gdf = gpd.read_file(fail)
    if gdf.crs and gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)
    
    # Lihtsusta geomeetriat (tolerance 0.001 kraadi ~ 100m täpsus, piisab kaardile)
    gdf['geometry'] = gdf['geometry'].simplify(tolerance=0.001, preserve_topology=True)
    
    valja = fail.replace('.geojson', '_small.geojson')
    gdf.to_file(valja, driver='GeoJSON')
    
    suurus_parast = os.path.getsize(valja) / 1024 / 1024
    print(f'{fail}: {suurus_enne:.1f}MB → {suurus_parast:.1f}MB')
    
    # Kui ikka liiga suur, lihtsusta rohkem
    if suurus_parast > 50:
        gdf['geometry'] = gdf['geometry'].simplify(tolerance=0.005, preserve_topology=True)
        gdf.to_file(valja, driver='GeoJSON')
        suurus_parast2 = os.path.getsize(valja) / 1024 / 1024
        print(f'  → Lihtsustati rohkem: {suurus_parast2:.1f}MB')

In [ ]:
# Kontrolli et nimed on ikka õiged
import json
for fail in geojson_failid:
    valja = fail.replace('.geojson', '_small.geojson')
    with open(valja, encoding='utf-8') as f:
        gj = json.load(f)
    props = list(gj['features'][0]['properties'].keys())
    nimed = [feat['properties'][props[0]] for feat in gj['features']]
    print(f'{valja}: {len(gj["features"])} piirkonda, property="{props[0]}"')
    print(f'  Näidis: {sorted(nimed)[:5]}')

In [ ]:
# Lae alla
for fail in geojson_failid:
    valja = fail.replace('.geojson', '_small.geojson')
    files.download(valja)